# DWBDA Assignment Spark Streaming

**Student:** RITIK DAS  
**Registration number:** 202400130  
**Semester:** B Tech CSE Data Science

This notebook streams simulated Teesta water-level sensor readings from JSON files. It uses checkpointing, watermark-based de-duplication, and a Parquet sink for fault-tolerant processing.

In [ ]:
%pip -q install pyspark==3.5.5

## Data source

The source is a directory of JSON files. Each file represents a micro-batch from five water-level sensors: Chungthang, Dikchu, Singtam, Rangpo, and Majitar. The generator deliberately adds duplicate readings and malformed JSON records to test the stream handling. Files are written to a temporary directory and atomically moved into the input directory so Spark never sees a partial file.

In [ ]:

import json
import random
import shutil
import time
from datetime import datetime, timedelta
from pathlib import Path
from threading import Event, Thread

# Set random seed
random.seed(42)

# Create project folders
base_folder = Path("teesta_stream")
input_folder = base_folder / "input"
temp_folder = base_folder / "tmp"
output_folder = base_folder / "output"
checkpoint_folder = base_folder / "checkpoint"

# Remove old folder if it already exists
shutil.rmtree(base_folder, ignore_errors=True)

# Create required folders
input_folder.mkdir(parents=True)
temp_folder.mkdir()
output_folder.mkdir()
checkpoint_folder.mkdir()

# List of monitoring stations
stations = ["Chungthang", "Dikchu", "Singtam", "Rangpo", "Majitar"]

# Starting date and time
current_time = datetime(2026, 11, 5, 10, 0, 0)

# Event to check whether data production is complete
producer_finished = Event()

# Dictionary to store metrics
metrics = {}


# Function to write data into a JSON file
def write_batch(file_name, lines):

    temp_file = temp_folder / f"{file_name}.json"
    input_file = input_folder / f"{file_name}.json"

    # Write data into temporary file
    temp_file.write_text("\n".join(lines), encoding="utf-8")

    # Move file to input folder
    temp_file.replace(input_file)


# Function to generate water-level readings
def produce_readings():

    current_time = datetime(2026, 11, 5, 10, 0, 0)

    previous_rows = []

    duplicate_count = 0
    bad_record_count = 0

    # Generate 40 batches of data
    for batch_id in range(40):

        rows = []

        # Generate readings for each station
        for station in stations:

            row = {
                "station": station,
                "id": batch_id,
                "time": str(current_time),
                "level": round(random.uniform(2, 6), 2)
            }

            rows.append(row)

        # Convert data into JSON format
        lines = [json.dumps(row) for row in rows]

        # Add duplicate records randomly
        if previous_rows and random.random() < 0.5:

            duplicate_row = random.choice(previous_rows)

            lines.append(duplicate_row)
            duplicate_count += 1

        # Add an incorrect record in some batches
        if batch_id % 8 == 5:

            lines.append("{bad record")
            bad_record_count += 1

        # Save the batch
        file_name = f"batch_{batch_id:02d}"

        write_batch(file_name, lines)

        # Store current rows for the next batch
        previous_rows = [json.dumps(row) for row in rows]

        # Increase time by one minute
        current_time += timedelta(minutes=1)

        time.sleep(1)

    # Create first finalizer record
    first_finalizer = {
        "station": "_finalizer_1",
        "id": -1,
        "time": str(current_time + timedelta(minutes=10)),
        "level": 0.0
    }

    # Create second finalizer record
    second_finalizer = {
        "station": "_finalizer_2",
        "id": -2,
        "time": str(current_time + timedelta(minutes=11)),
        "level": 0.0
    }

    # Save first finalizer
    write_batch("finalizer_1", [json.dumps(first_finalizer)])

    time.sleep(1)

    # Save second finalizer
    write_batch("finalizer_2", [json.dumps(second_finalizer)])

    # Store final statistics
    metrics.update({
        "files": 42,
        "duplicates": duplicate_count,
        "malformed": bad_record_count
    })

    # Mark data generation as completed
    producer_finished.set()


print("Generator is ready. The streaming cell starts it after the query is running.")

## Structured Streaming query

The file source reads one file per micro-batch. Invalid records are discarded, while valid records are de-duplicated by `station` and `id` inside a five-minute event-time watermark. The query computes the average and maximum water level for each five-minute window and station. Two finalizer records advance the watermark so the final regular windows can be written.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, max, window
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

# Create Spark session
spark = SparkSession.builder \
    .appName("TeestaWaterLevelStream") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Set log level
spark.sparkContext.setLogLevel("ERROR")

# Define the structure of incoming data
schema = StructType([
    StructField("station", StringType(), True),
    StructField("id", IntegerType(), True),
    StructField("time", TimestampType(), True),
    StructField("level", DoubleType(), True),
    StructField("_corrupt_record", StringType(), True)
])

# Read streaming data from input folder
stream = spark.readStream \
    .schema(schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .json(str(input_dir))

# Remove incorrect and invalid records
cleaned_data = stream.filter(
    col("station").isNotNull() &
    col("_corrupt_record").isNull()
)

# Set watermark and remove duplicate records
cleaned_data = cleaned_data \
    .withWatermark("time", "5 minutes") \
    .dropDuplicatesWithinWatermark(["station", "id"])

# Group data into 5-minute windows
grouped_data = cleaned_data.groupBy(
    window("time", "5 minutes"),
    "station"
)

# Calculate average and maximum water level
result = grouped_data.agg(
    avg("level").alias("avg_level"),
    max("level").alias("max_level")
)

# Start streaming query
query = result.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", str(output_dir)) \
    .option("checkpointLocation", str(checkpoint_dir)) \
    .trigger(processingTime="1 second") \
    .start()

# Start the data producer in a separate thread
producer = Thread(
    target=produce_readings,
    daemon=True
)

producer.start()

# Set maximum waiting time to 90 seconds
deadline = time.monotonic() + 90

# Wait until all files are processed
while time.monotonic() < deadline:

    commits_folder = checkpoint_dir / "commits"

    if commits_folder.exists():
        completed_batches = sum(
            file.name.isdigit()
            for file in commits_folder.glob("*")
        )
    else:
        completed_batches = 0

    # Check whether data production and processing are complete
    if producer_finished.is_set() and completed_batches >= 42:
        break

    time.sleep(1)

else:
    query.stop()
    raise TimeoutError("The stream did not process all input files.")

# Stop the streaming query
query.stop()

# Wait for producer thread to finish
producer.join()

# Display final results
print(
    f"Streaming query completed: "
    f"{metrics['files']} files, "
    f"{metrics['duplicates']} duplicates, "
    f"{metrics['malformed']} malformed records."
)

## Output

There are eight five-minute windows and five stations, so the clean completed stream produces 40 aggregate rows. Duplicate and malformed inputs do not create extra output rows.

In [ ]:
# Read the saved Parquet data
saved_data = spark.read.parquet(str(output_dir))

# Remove finalizer records
saved_data = saved_data.filter(
    ~col("station").startswith("_finalizer")
)

# Count the total number of saved records
total_windows = saved_data.count()

print("Windows saved:", total_windows)

# Display the saved data in sorted order
saved_data.orderBy(
    "window",
    "station"
).show(50, truncate=False)

## Fault tolerance and checkpointing

The checkpoint directory stores source progress, sink commits, and state for watermark de-duplication. If the query is restarted with the same checkpoint path, Spark resumes from the last committed micro-batch. The JSON files remain in the input directory, so incomplete work can be replayed. The Parquet sink keeps a metadata log and avoids rewriting committed batches.

The five-minute watermark limits the amount of de-duplication state retained. Duplicate readings use the same station and id, so `dropDuplicatesWithinWatermark` removes them even after a query restart. Malformed JSON maps to `_corrupt_record` and is filtered before aggregation.

In [ ]:
# Restart the streaming query
recovery_query = result.writeStream \
    .outputMode("append") \
    .format("parquet") \
    .option("path", str(output_dir)) \
    .option("checkpointLocation", str(checkpoint_dir)) \
    .trigger(availableNow=True) \
    .start()

# Wait until the streaming process is completed
recovery_query.awaitTermination()

# Read the saved data after restarting
after_restart = spark.read.parquet(str(output_dir)).count()

# Display the total number of windows
print("Windows after restart:", after_restart)

# Stop the Spark session
spark.stop()

## Conclusion

This implementation uses Spark Structured Streaming with a replayable file source, atomic file delivery, watermark-based de-duplication, checkpoint recovery, and fault-tolerant Parquet output.